# 🤝 Agent Handoffs in LangGraph

## Learning Objectives
In this notebook, you will learn:
1. **Handoff State Design** - How to design a shared `TypedDict` state that carries messages, the active agent, and context across handoffs
2. **Structured Handoff Decisions** - How to use a Pydantic schema with `with_structured_output` to make an LLM emit a reliable routing decision
3. **Multi-Agent Routing** - How to wire a triage node to specialist nodes using `add_conditional_edges`
4. **Context Passing** - How to summarize and forward context from a triage agent to the specialist that receives the handoff

## Prerequisites
- Familiarity with LangGraph's `StateGraph`, nodes, and conditional edges
- Understanding of Pydantic models and `with_structured_output`
- An `OPENAI_API_KEY` set in a `.env` file at the project root

> Converted from `03_agent_handoffs.py` - part of **04 Multi Agent Systems**.

---
## 🔧 Part 1: Environment Setup

We load environment variables from `.env` and initialize the LLM that every agent node in this notebook will share.

In [ ]:
# ============================================================================
# ENVIRONMENT SETUP: Imports & LLM Initialization
# ============================================================================
import operator
from typing import Literal

from dotenv import load_dotenv
from pydantic import BaseModel, Field
from typing_extensions import Annotated, TypedDict

from langchain_core.messages import AIMessage, BaseMessage, HumanMessage, SystemMessage
from langchain_openai import ChatOpenAI
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages

load_dotenv()

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

print(f"🤖 LLM initialized: {llm.model_name}")

---
## 🗂️ Part 2: State & Decision Schemas

Before building the graph, we define two data structures: the shared `HandoffState` that flows through every node, and the `HandoffDecision` schema the triage agent fills in to decide where a customer should go.

### Key Concepts:
- **Shared State**: All nodes read from and write to the same `HandoffState`, which is how context survives a handoff
- **Structured Output**: `HandoffDecision` constrains the LLM to a fixed set of valid destinations so routing logic never has to parse free text

### `HandoffState` - Shared Graph State

In [ ]:
# ============================================================================
# HANDOFF STATE: Shared State Across All Agents
# ============================================================================
class HandoffState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]
    current_agent: str
    handoff_reason: str
    context_summary: str

### `HandoffDecision` - Structured Routing Output

In [ ]:
# ============================================================================
# HANDOFF DECISION: Structured Output Schema for Triage
# ============================================================================
class HandoffDecision(BaseModel):
    handoff_to: Literal["sales", "support", "billing", "stay", "end"] = Field(
        description="Which agent to hand off to"
    )
    reason: str = Field(description="Reason for handoff")
    context: str = Field(description="Key context to pass to next agent")

---
## 🏗️ Part 3: Building the Customer Service Handoff System

This section assembles the full graph: a `triage` node that classifies the customer's request, three specialist nodes (`sales`, `support`, `billing`), and a routing function that connects triage to the right specialist via `add_conditional_edges`.

> **Key Insight**: The triage agent can also resolve simple questions itself (`handoff_to="end"`) without ever handing off to a specialist - a handoff only happens when specialist expertise is actually needed.

### `create_customer_service_system` - Graph Factory

In [ ]:
# ============================================================================
# CREATE_CUSTOMER_SERVICE_SYSTEM: Assemble the Triage + Specialist Graph
# ============================================================================
def create_customer_service_system():
    def triage_agent(state: HandoffState) -> dict:
        """Initial triage to route customer."""
        system = """You are a customer service triage agent. Your job is to:
        1. Understand the customer's need
        2. Route to the appropriate specialist:
           - sales: Product questions, purchases, upgrades
           - support: Technical issues, bugs, how-to questions
           - billing: Payments, invoices, refunds
           - end: Simple questions you can answer directly

        Analyze the customer's message and decide where to route them."""

        handoff_llm = llm.with_structured_output(HandoffDecision)
        messages = [SystemMessage(content=system)] + state["messages"]
        decision = handoff_llm.invoke(messages)

        if decision.handoff_to == "end":
            # Answer directly
            response = llm.invoke(
                [
                    SystemMessage(
                        content="Provide a brief, helpful response to the customer."
                    ),
                    *state["messages"],
                ]
            )
            return {
                "messages": [AIMessage(content=f"[Triage] {response.content}")],
                "current_agent": "end",
            }

        return {
            "current_agent": decision.handoff_to,
            "handoff_reason": decision.reason,
            "context_summary": decision.context,
            "messages": [
                AIMessage(
                    content=f"[Triage] Transferring to {decision.handoff_to}: {decision.reason}"
                )
            ],
        }

    def sales_agent(state: HandoffState) -> dict:
        """Sales specialist."""
        system = f"""You are a sales specialist. Context from triage: {state.get('context_summary', 'None')}

            Help the customer with product questions and purchases.
            Be helpful and informative, not pushy."""

        response = llm.invoke([SystemMessage(content=system), *state["messages"]])
        return {
            "messages": [AIMessage(content=f"[Sales] {response.content}")],
            "current_agent": "sales_complete",
        }

    def support_agent(state: HandoffState) -> dict:
        """Technical support specialist."""
        system = f"""You are a technical support specialist. Context from triage: {state.get('context_summary', 'None')}

        Help the customer with technical issues.
        Be patient and provide step-by-step guidance."""

        response = llm.invoke([SystemMessage(content=system), *state["messages"]])
        return {
            "messages": [AIMessage(content=f"[Support] {response.content}")],
            "current_agent": "support_complete",
        }

    def billing_agent(state: HandoffState) -> dict:
        """Billing specialist."""
        system = f"""You are a billing specialist. Context from triage: {state.get('context_summary', 'None')}

        Help the customer with billing questions.
        Be clear about policies and next steps."""

        response = llm.invoke([SystemMessage(content=system), *state["messages"]])
        return {
            "messages": [AIMessage(content=f"[Billing] {response.content}")],
            "current_agent": "billing_complete",
        }

    def route_from_triage(state: HandoffState) -> str:
        agent = state["current_agent"]
        if agent in ["sales", "support", "billing"]:
            return agent
        return "end"

    graph = StateGraph(HandoffState)
    graph.add_node("triage", triage_agent)
    graph.add_node("sales", sales_agent)
    graph.add_node("support", support_agent)
    graph.add_node("billing", billing_agent)

    graph.add_edge(START, "triage")
    graph.add_conditional_edges(
        "triage",
        route_from_triage,
        {"sales": "sales", "support": "support", "billing": "billing", "end": END},
    )
    graph.add_edge("sales", END)
    graph.add_edge("support", END)
    graph.add_edge("billing", END)

    return graph.compile()

print("✅ Customer service handoff graph factory defined: create_customer_service_system()")

### `demo_handoffs` - Running the Handoff Demo

This helper builds the graph and runs it against a handful of sample customer messages, printing which agent responded to each one so you can see the routing decisions in action.

In [ ]:
# ============================================================================
# DEMO_HANDOFFS: Exercise the Graph Against Sample Customer Messages
# ============================================================================
def demo_handoffs():
    """Demo customer service handoffs."""
    agent = create_customer_service_system()

    print("Customer Service Handoff Demo:\n")

    queries = [
        "My app keeps crashing when I try to upload photos",
        "I want to upgrade to the premium plan",
        "I was charged twice for my subscription",
        "What time do you close?",
    ]

    for query in queries:
        print(f"Customer: {query}")
        result = agent.invoke(
            {
                "messages": [HumanMessage(content=query)],
                "current_agent": "",
                "handoff_reason": "",
                "context_summary": "",
            }
        )
        for msg in result["messages"]:
            if isinstance(msg, AIMessage):
                print(f"  {msg.content[:150]}...")
        print("-" * 50)

---
## ▶️ Part 4: Run the Demo

The original `__main__` guard, kept verbatim. Jupyter sets `__name__` to `"__main__"`, so this cell runs as-is - uncomment a line to run that demo.

In [ ]:
# ============================================================================
# RUN: Execute the Demo When Run as a Script
# ============================================================================
if __name__ == "__main__":
    demo_handoffs()

---
## 📝 Summary

In this notebook, we learned:

### 1. State & Schema Design
- **`HandoffState`**: A shared `TypedDict` that carries the message history plus `current_agent`, `handoff_reason`, and `context_summary` across nodes
- **`HandoffDecision`**: A Pydantic schema paired with `with_structured_output` so the triage LLM returns a validated routing decision instead of free text

### 2. Routing & Handoff Logic
- **`create_customer_service_system()`**: Builds a `StateGraph` with a `triage` node and three specialists (`sales`, `support`, `billing`), connected via `add_conditional_edges`
- The triage agent can resolve simple questions itself (`handoff_to="end"`) without handing off at all
- Context gathered during triage (`context_summary`) is passed along so the specialist doesn't have to re-ask the customer

### 3. Demo & Execution
- **`demo_handoffs()`**: Runs the graph against representative support, sales, billing, and general queries and prints which agent handled each one

### Next Steps
- Explore how to add a **handoff-back** path so a specialist can return a customer to triage if they were misrouted
- Look at persisting `HandoffState` across turns with a checkpointer so a customer's context survives multiple messages